<a href="https://colab.research.google.com/github/srimathichinnasamy/next-word-prediction/blob/main/next_word_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorflow pandas numpy scikit-learn

In [5]:


# 1. Load dataset
sentences = []

if os.path.exists("corpus.txt"):
    with open("corpus.txt", "r", encoding="utf-8") as file:
        sentences = file.readlines()
else:
    csv_files = glob.glob("*.csv")
    if csv_files:
        df = pd.read_csv(csv_files[0])
        # Find a suitable text column
        possible_columns = [
            "text", "content", "sentence", "title", "statement", "article"
        ]
        text_column = next(
            (c for c in possible_columns if c in df.columns.str.lower()),
            None
        )
        if text_column is None:
            text_column = df.select_dtypes(
                include=["object"]
            ).columns[0]
        actual_column = next(
            c for c in df.columns if c.lower() == text_column
        )
        sentences = df[actual_column].dropna().astype(str).tolist()
    else:
        print("No local corpus.txt or CSV file found. Downloading a sample text (Shakespeare's Romeo and Juliet) from Project Gutenberg...")
        url = "https://www.gutenberg.org/cache/epub/1513/pg1513.txt"
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as response:
                text_data = response.read().decode('utf-8')
                sentences = text_data.split('\n')
        except Exception as e:
            print(f"Failed to download sample corpus: {e}")
            raise FileNotFoundError("Please place corpus.txt or a dataset CSV file in this folder.")

# Use approximately 10,000 sentences
sentences = sentences[:10000]

# 2. Clean text
clean_sentences = []

for sentence in sentences:
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-z\s']", " ", sentence)
    sentence = re.sub(r"\s+", " ", sentence).strip()

    if len(sentence.split()) >= 2:
        clean_sentences.append(sentence)

print("Number of sentences:", len(clean_sentences))

# 3. Tokenization and vocabulary
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(clean_sentences)

vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary size:", vocab_size)

# 4. Generate input-target sequences
sequences = []

for sentence in clean_sentences:
    tokens = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(2, len(tokens) + 1):
        sequences.append(tokens[:i])

if not sequences:
    raise ValueError("No sequences generated. Check if the text processing yields non-empty sequences.")

max_length = max(len(seq) for seq in sequences)
max_length = min(max_length, 30)

# Keep the most recent words as input
sequences = [
    seq[-max_length:] for seq in sequences
]

sequences = pad_sequences(
    sequences, maxlen=max_length, padding="pre"
)

X = sequences[:, :-1]
y = sequences[:, -1]

print("Input shape:", X.shape)
print("Target shape:", y.shape)

# 5. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 6. Build LSTM model
model = Sequential([
    Input(shape=(max_length - 1,)),
    Embedding(vocab_size, 64, mask_zero=True),
    LSTM(64),
    Dropout(0.2),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# 7. Train the model
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

# 8. Evaluate top-1 and top-5 accuracy
predictions = model.predict(X_test, verbose=0)

top1 = np.mean(
    np.argmax(predictions, axis=1) == y_test
)

top5_indices = np.argsort(predictions, axis=1)[:, -5:]
top5 = np.mean(
    np.any(top5_indices == y_test[:, None], axis=1)
)

print("\nTop-1 Accuracy:", round(top1 * 100, 2), "%")
print("Top-5 Accuracy:", round(top5 * 100, 2), "%")

# 9. Predict the next word
def predict_next_word(phrase):
    phrase = phrase.lower()
    phrase = re.sub(r"[^a-z\s']", " ", phrase)
    phrase = re.sub(r"\s+", " ", phrase).strip()

    sequence = tokenizer.texts_to_sequences([phrase])[0]
    sequence = sequence[-(max_length - 1):]

    padded = pad_sequences(
        [sequence], maxlen=max_length - 1, padding="pre"
    )

    probabilities = model.predict(padded, verbose=0)[0]
    top_words = np.argsort(probabilities)[-5:][::-1]

    print("\nInput:", phrase)
    print("Top predictions:")

    for word_id in top_words:
        word = tokenizer.index_word.get(word_id, "<unknown>")
        print(word, "-", round(float(probabilities[word_id]) * 100, 2), "%")

# 10. Test five phrases
test_phrases = [
    "the way to",
    "this is a",
    "I want to",
    "there is a",
    "one of the"
]

for phrase in test_phrases:
    predict_next_word(phrase)

No local corpus.txt or CSV file found. Downloading a sample text (Shakespeare's Romeo and Juliet) from Project Gutenberg...
Number of sentences: 3660
Vocabulary size: 3993
Input shape: (25469, 17)
Target shape: (25469,)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 17, 64)         │       255,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3993)           │       259,545 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 548,121 (2.09 MB)

 Trainable params: 548,121 (2.09 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 11s 32ms/step - accuracy: 0.0286 - loss: 7.1553 - val_accuracy: 0.0358 - val_loss: 6.7365
Epoch 2/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step - accuracy: 0.0303 - loss: 6.5123 - val_accuracy: 0.0358 - val_loss: 6.7264
Epoch 3/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.0313 - loss: 6.3935 - val_accuracy: 0.0427 - val_loss: 6.7546
Epoch 4/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.0343 - loss: 6.3192 - val_accuracy: 0.0432 - val_loss: 6.7968
Epoch 5/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 9s 29ms/step - accuracy: 0.0383 - loss: 6.2551 - val_accuracy: 0.0447 - val_loss: 6.8055
Epoch 6/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 9s 32ms/step - accuracy: 0.0410 - loss: 6.1869 - val_accuracy: 0.0442 - val_loss: 6.8276
Epoch 7/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.0459 - loss: 6.1182 - val_accuracy: 0.0530 - val_loss: 6.8116
Epoch 8/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 9s 32ms/step - accuracy: 0.0545 - loss: 6.0405 - val_a